# Upscaling of a Spline
We display in thick <span style="color:#e0e0e0">**light gray**</span> the realization of a random periodic polynomial spline $f_{0}$ of a specified period, degree $n_{0}$, and delay $\delta x_{0};$ the red stemlines indicate the extent of one period, while the <span style="color:#a0a0a0">**dark gray**</span> dots indicate the knots of $f_{0}.$ We then plot in thinner <span style="color:#1f77b4">**blue**</span> its projection $f$ of degree $n$ and delay $\delta x.$

In [22]:
# Load the required libraries
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

import splinekit as sk # This library

# Setup
max_period = 15 # Maximal period
max_degree = 9 # Maximal spline degree
max_delay = 3.0 # Maximal absolute delay
max_magnif = 5 # Maximal magnification factor

# Initialize the generator of random numbers
rng = np.random.default_rng()

# Initial random periodic cubic spline
f = sk.PeriodicSpline1D.from_spline_coeff(rng.standard_normal(6), degree = 3)

# Plot
def update_plot (
    period = 6,
    degree = 3,
    delay = 0.0,
    magnif = 2,
    x = 0.0,
    xm = 0.0
):
    global f

    # Update of the spline
    if f.period != period:
        f = sk.PeriodicSpline1D.from_spline_coeff(
            rng.standard_normal(period),
            degree = f.degree
        )
    f.degree = degree
    f.delay = delay

    # Upscaling
    fm = f.upscaled(magnification = magnif)

    # Values
    fx = f.at(x)
    fmx = fm.at(xm)

    # Plots
    (fig, (ax1, ax2, ax3, ax4)) = plt.subplots(
        nrows = 4,
        sharey = True,
        height_ratios = [5, 1, 5, 1]
    )
    # Caption
    ax2.spines[:].set_color("None")
    ax2.tick_params(bottom = False, labelbottom = False, left = False, labelleft = False)
    ax2.text(0.0, 0.0, r"$f({:.4f})={:.4f}$".format(x, fx))
    ax4.spines[:].set_color("None")
    ax4.tick_params(bottom = False, labelbottom = False, left = False, labelleft = False)
    ax4.text(
        0.0,
        0.0,
        r"$f_{{{}}}({:.4f})=f_{{{}}}({}\times{:.4f})={:.4f}$"
            .format(magnif, xm, magnif, magnif, x, fmx)
    )
    # Plot of the spline being magnified
    f.plot((fig, ax1), plotpoints = 200 + 1)
    # Integration bounds
    (markerline, _, _) = ax1.stem([x], [fx], "ko")
    markerline.set_markersize(11)
    # Plot of the upscaled spline
    fm.plot(
        (fig, ax3),
        plotdomain = sk.interval.Open((-magnif, magnif * (period + 1))),
        plotpoints = 200 + 1
    )
    (markerline, _, _) = ax3.stem([xm], [fmx], "ko")
    markerline.set_markersize(11)

    # Final display
    plt.show()

# Interaction
period_widget = widgets.IntSlider(min = 1, max = max_period, value = f.period)
magnif_widget = widgets.IntSlider(min = 1, max = max_magnif, value = 2)
x_widget = widgets.FloatSlider(
    min = -1.0,
    max = f.period + 1.0,
    value = 0.0,
    continuous_update = False
)
xm_widget = widgets.FloatSlider(
    min = -2.0,
    max = 2 * (f.period + 1.0),
    value = 0.0,
    continuous_update = False
)
def update_period (
    change
):
    x_widget.max = change.new + 1.0
    xm_widget.max = magnif_widget.value * (change.new + 1.0)
period_widget.observe(update_period, "value")
def update_magnif (
    change
):
    xm_widget.min = 0.0 - change.new
    xm_widget.max = change.new * (period_widget.value + 1.0)
    xm_widget.value = x_widget.value * change.new
magnif_widget.observe(update_magnif, "value")
def update_x (
    change
):
    xm_widget.value = change.new * magnif_widget.value
x_widget.observe(update_x, "value")
def update_xm (
    change
):
    x_widget.value = change.new / magnif_widget.value
xm_widget.observe(update_xm, "value")
widgets.interactive(
    update_plot,
    period = period_widget,
    degree = (0, max_degree),
    delay = (-max_delay, max_delay),
    magnif = magnif_widget,
    x = x_widget,
    xm = xm_widget
)


interactive(children=(IntSlider(value=6, description='period', max=15, min=1), IntSlider(value=3, description=…